In [1]:
import torch
from ultralytics import YOLO

# 1. Check CUDA safely
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not found. Training will happen on CPU (this is okay for YOLOv5 Nano).")

model = YOLO('yolov3-tiny.pt') # Load the "Old Reliable"

CUDA available: True
Using GPU: Quadro P1000
PRO TIP 💡 Replace 'model=yolov3-tiny.pt' with new 'model=yolov3-tinyu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



In [2]:
# We use the same 240x240 size defined in your preprocessing script
model.train(
    data='/home/wiebe/Desktop/Robotic/AE4317/testing_nn/yolo/data.yaml', 
    epochs=50, 
    imgsz=240, 
    device=0, 
    workers=4  # Helps your CPU feed the GPU faster
)

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.5.1 CUDA:0 (Quadro P1000, 4032MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/wiebe/Desktop/Robotic/AE4317/testing_nn/yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=240, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov3-tiny.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

ImportError: cannot import name 'ERR_IGNORE' from 'numpy.core.umath' (/home/wiebe/miniconda3/envs/bebop_vision/lib/python3.10/site-packages/numpy/core/umath.py)

In [4]:
from ultralytics import YOLO

# 1. Load your newly trained YOLOv3-Tiny weights
model = YOLO('runs/detect/train11/weights/best.pt') 

# 2. Export to ONNX
# We use 256 for alignment and opset 10 for 'Old Math' compatibility
model.export(format='onnx', imgsz=256, opset=9, simplify=True)

print("SUCCESS: YOLOv3-Tiny exported for OpenCV 4.5.4")

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.5.1 CPU (Intel Core i7-8750H 2.20GHz)
YOLOv3-tiny summary (fused): 34 layers, 12,128,692 parameters, 0 gradients, 18.9 GFLOPs

PyTorch: starting from 'runs/detect/train11/weights/best.pt' with input shape (1, 3, 256, 256) BCHW and output shape(s) (1, 6, 320) (23.2 MB)

ONNX: starting export with onnx 1.20.1 opset 9...


/home/wiebe/miniconda3/envs/bebop_vision/lib/python3.10/site-packages/torch/onnx/symbolic_helper.py:795: UserWarning: You are trying to export the model with onnx:Upsample for ONNX opset version 9. This operator might cause results to not match the expected results by PyTorch.
ONNX's Upsample/Resize operator did not match Pytorch's Interpolation until opset 11. Attributes to determine how to transform the input were added in onnx:Resize in opset 11 to support Pytorch's behavior (like coordinate_transformation_mode and nearest_mode).
We recommend using opset 11 and above for models using this operator.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.87...
ONNX: export success ✅ 0.7s, saved as 'runs/detect/train11/weights/best.onnx' (46.3 MB)

Export complete (0.9s)
Results saved to /home/wiebe/Desktop/Robotic/AE4317/testing_nn/yolo/runs/detect/train11/weights
Predict:         yolo predict task=detect model=runs/detect/train11/weights/best.onnx imgsz=256 
Validate:        yolo val task=detect model=runs/detect/train11/weights/best.onnx imgsz=256 data=/home/wiebe/Desktop/Robotic/AE4317/testing_nn/yolo/data.yaml  
Visualize:       https://netron.app
SUCCESS: YOLOv3-Tiny exported for OpenCV 4.5.4


In [1]:
from ultralytics import YOLO

# 1. Load your trained best.pt
model = YOLO('runs/detect/train12/weights/best.pt')

# 2. Export to ONNX
# imgsz must be 256 to match your current architecture
# opset=12 is the 'golden version' for compatibility with onnx2c
model.export(format='onnx', imgsz=256, opset=13, simplify=True)

print("Export Complete: best.onnx is ready for conversion.")

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.5.1 CPU (Intel Core i7-8750H 2.20GHz)
YOLOv3-tiny summary (fused): 34 layers, 12,128,692 parameters, 0 gradients, 18.9 GFLOPs

PyTorch: starting from 'runs/detect/train12/weights/best.pt' with input shape (1, 3, 256, 256) BCHW and output shape(s) (1, 6, 320) (23.2 MB)

ONNX: starting export with onnx 1.20.1 opset 13...


/home/wiebe/miniconda3/envs/bebop_vision/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647327489/work/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/wiebe/miniconda3/envs/bebop_vision/lib/python3.10/site-packages/torch/onnx/utils.py:663: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647327489/work/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/home/wiebe/miniconda3/envs/bebop_vision/lib/python3.10/site-packages/torch/onnx/utils.py:1186: UserWarning: Constant folding - Only steps=1 can be constant fol

ONNX: slimming with onnxslim 0.1.87...
ONNX: export success ✅ 1.6s, saved as 'runs/detect/train12/weights/best.onnx' (46.3 MB)

Export complete (2.1s)
Results saved to /home/wiebe/Desktop/Robotic/AE4317/testing_nn/yolo/runs/detect/train12/weights
Predict:         yolo predict task=detect model=runs/detect/train12/weights/best.onnx imgsz=256 
Validate:        yolo val task=detect model=runs/detect/train12/weights/best.onnx imgsz=256 data=/home/wiebe/Desktop/Robotic/AE4317/testing_nn/yolo/data.yaml  
Visualize:       https://netron.app
Export Complete: best.onnx is ready for conversion.


In [5]:
import cv2
import numpy as np

# Load the files you just made
net = cv2.dnn.readNetFromDarknet('yolov3-tiny.cfg', 'best.weights')
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

# Create a fake image (or load a real one from your dataset)
test_img = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
blob = cv2.dnn.blobFromImage(test_img, 1.0/255.0, (256, 256), swapRB=True)

net.setInput(blob)
outs = net.forward(net.getUnconnectedOutLayersNames())

# Find the max confidence in the output
max_conf = 0
for out in outs:
    for detection in out:
        conf = detection[4]
        if conf > max_conf:
            max_conf = conf

print(f"Max Confidence Found: {max_conf:.6f}")
if max_conf == 0:
    print("ERROR: The weights file is still broken/scrambled.")
else:
    print("SUCCESS: The files are good. The issue is in the C++ preprocessing.")

Max Confidence Found: 0.000000
ERROR: The weights file is still broken/scrambled.
